# H19a: Partial SV Equalization Creates Spurious Saddle Points

## Motivation and Scientific Context

This experiment addresses one of the most surprising findings from H3b: **partial singular value (SV) equalization is not merely weaker than full equalization -- it is actively destructive**, performing *worse* than plain SGD. This is not a "less is less" result; it is a "less is *harmful*" result, which demands a geometric explanation.

### The Frankensteinian Geometry Hypothesis

When we apply Muon-style optimization, we equalize the singular values of per-layer gradient matrices. Full equalization (k = dim) replaces the gradient with its polar factor $G = U V^T$, producing an **isotropic** update geometry: every direction in weight space receives equal effective step size. SGD (k = 0) preserves the **original anisotropic** geometry of the gradient. Both are internally consistent.

**Partial equalization** (0 < k < dim) creates a geometric chimera:
- The **top-k** singular directions are forced toward uniformity (equal singular values)
- The **remaining** (dim - k) directions retain their original, potentially very different magnitudes

This mismatch creates **conflicting curvature signals**. The equalized subspace "wants" the optimizer to explore uniformly, while the non-equalized subspace pulls toward the original landscape's anisotropy. At the boundary between these two regimes, the effective Hessian can develop **negative eigenvalues** -- directions along which the step actually *increases* loss curvature in a destabilizing way. These are **spurious saddle points** that do not exist in either the pure SGD or pure Muon landscape.

### What This Experiment Measures

We construct a 3-layer deep linear network (4x4 matrices, 48 total parameters) where the **full 48x48 Hessian** is computationally tractable via finite differences. After 50 warmup SGD steps (to reach a non-trivial point in the landscape), we:

1. Compute the Hessian **before** the step (baseline saddle structure)
2. Take **one step** with partial SV equalization at level k (per layer)
3. Compute the Hessian **after** the step
4. Count negative eigenvalues and measure the most negative eigenvalue

### Key Prediction

| k (per layer) | Expected saddle directions | Interpretation |
|---|---|---|
| k=0 (SGD) | Baseline | Consistent anisotropic geometry |
| k=1,2,3 | **More** than baseline | Frankensteinian mismatch |
| k=4 (full Muon) | Baseline or fewer | Consistent isotropic geometry |

The "U-shape" (or "saddle hump") in negative eigenvalue count as a function of k is the signature of spurious saddle creation by partial equalization.

---
## Section 1: Imports and Environment Setup

In [ ]:
"""
H19a: Partial SV Equalization Creates Spurious Saddle Points
==============================================================

FROM H3b: k<32 is WORSE than SGD (not just weaker -- actively destructive).
This is the most surprising result: partial equalization HURTS.

HYPOTHESIS:
  Partial SV equalization (top-k SVs equalized, rest proportional) creates
  an inconsistent geometry: the equalized SVs want to explore uniformly
  while the non-equalized SVs pull toward the original anisotropic landscape.
  This mismatch introduces negative Hessian eigenvalues (saddle points) along
  directions that mix equalized and non-equalized SVs.

  Full equalization (k=dim) avoids this because ALL directions are treated
  uniformly. k=0 (SGD) avoids it because the landscape is consistent.
  The half-measure creates a Frankensteinian geometry.

PROTOCOL:
  3-layer deep linear 4x4 (48 params, full Hessian tractable).
  At step 50 (after some training), for each k in {0, 4, 8, 16, 32}:
    1. Compute the Hessian at the current point
    2. Apply one partial-SV-equalized step
    3. Compute the Hessian at the new point
    4. Count negative eigenvalues (saddle directions)
    5. Measure the minimum eigenvalue (most negative = sharpest saddle)
  Compare negative eigenvalue counts across k.

KEY PREDICTION:
  Intermediate k (4, 8, 16) produce MORE negative Hessian eigenvalues
  than k=0 (SGD) or k=32 (full Muon).
"""

In [ ]:
import numpy as np
import os

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))

---
## Section 2: Experimental Constants

The network architecture is deliberately small: 3 layers of 4x4 matrices yield 48 total parameters. This makes the full Hessian (a 48x48 matrix) computable via finite differences in reasonable time, while still exhibiting the non-trivial saddle structure of deep linear networks.

**Why 50 warmup steps?** We need the network to have moved away from its near-identity initialization into a region where the loss landscape has meaningful curvature. Too few warmup steps and the Hessian is nearly flat; too many and the network may have converged past the interesting saddle regions.

**Why finite difference epsilon = 1e-5?** This balances numerical precision (smaller eps = more accurate gradient of gradient) against floating-point noise (too small and subtraction cancellation dominates). For 64-bit floats, 1e-5 is near the sweet spot for second-derivative estimation.

In [ ]:
DIM = 4
N_LAYERS = 3
N_PARAMS = N_LAYERS * DIM * DIM  # 48
WARMUP_STEPS = 50
NUM_SEEDS = 5
MOMENTUM = 0.9
LR = 0.01
FD_EPS = 1e-5

print("=== Experimental Constants ===")
print(f"  Network: {N_LAYERS}-layer deep linear, each layer {DIM}x{DIM}")
print(f"  Total parameters: {N_PARAMS}")
print(f"  Hessian size: {N_PARAMS}x{N_PARAMS} = {N_PARAMS**2} entries")
print(f"  Warmup: {WARMUP_STEPS} SGD steps (momentum={MOMENTUM}, lr={LR})")
print(f"  Finite difference epsilon: {FD_EPS}")
print(f"  Number of random seeds: {NUM_SEEDS}")
print(f"  Each seed requires {N_PARAMS} Hessian columns x {len([0,1,2,3,4])+1} k-values = {N_PARAMS * 6} gradient evaluations")

### k-values: The Independent Variable

For a 4x4 matrix, the SVD produces exactly 4 singular values. So the per-layer k ranges from 0 (no equalization, pure SGD) to 4 (full equalization, polar factor / full Muon). The intermediate values k=1, 2, 3 are where the "Frankensteinian" mismatch geometry should emerge.

Note the distinction between `K_VALUES` (originally defined for a different parameterization) and `K_PER_LAYER` which is what actually controls the experiment -- it indexes per-layer SVD equalization depth.

In [ ]:
K_VALUES = [0, 2, 4, 8, 16]  # 0=SGD, 16=full (DIM=4, so 16=N_PARAMS/3 per layer)
# For DIM=4, per-layer matrix is 4x4, SVD has 4 SVs.
# k values for equalization: 0 (none), 1, 2, 3, 4 (all) per layer
K_PER_LAYER = [0, 1, 2, 3, 4]

print("=== k-value Parameterization ===")
print(f"  Per-layer SVD dimension: min(DIM, DIM) = {min(DIM, DIM)} singular values")
print(f"  K_PER_LAYER = {K_PER_LAYER}")
for k in K_PER_LAYER:
    if k == 0:
        desc = "pure SGD gradient (no SV modification)"
    elif k == DIM:
        desc = "full polar factor (all SVs -> 1, Muon-equivalent)"
    else:
        desc = f"top-{k} SVs equalized to their mean, bottom-{DIM-k} rescaled"
    print(f"    k={k}: {desc}")

---
## Section 3: Network Utilities (pack/unpack/forward)

The deep linear network is parameterized as a list of 3 weight matrices $W_1, W_2, W_3$, each $4 \times 4$. For Hessian computation via finite differences, we need a flat parameter vector $\theta \in \mathbb{R}^{48}$. The `pack` and `unpack` functions handle this bijection.

The forward pass computes $\hat{Y} = W_3 W_2 W_1 X$ -- a simple chain of matrix multiplications with no nonlinearities. Despite this linearity, the loss landscape is **non-convex** due to the multiplicative interaction between layers, creating the saddle point structure we are probing.

In [ ]:
def pack(Ws):
    return np.concatenate([W.ravel() for W in Ws])

In [ ]:
def unpack(theta):
    Ws = []
    idx = 0
    for _ in range(N_LAYERS):
        Ws.append(theta[idx:idx + DIM*DIM].reshape(DIM, DIM))
        idx += DIM * DIM
    return Ws

In [ ]:
def forward(Ws, X):
    out = X.copy()
    for W in Ws:
        out = W @ out
    return out

---
## Section 4: Loss and Gradient Computation

The loss is the standard mean squared error: $\mathcal{L} = \frac{1}{2N} \sum_i \| \hat{y}_i - y_i \|^2$.

The gradient is computed analytically via backpropagation through the linear layers. For layer $l$, the gradient is $\nabla_{W_l} \mathcal{L} = \delta_l \cdot a_{l-1}^T$, where $\delta_l$ is the error signal propagated backward and $a_{l-1}$ is the activation from the layer below. The function returns both the packed gradient vector (for Hessian computation) and the per-layer gradient matrices (for SV equalization).

In [ ]:
def loss_fn(theta, X, Y):
    Ws = unpack(theta)
    pred = forward(Ws, X)
    return 0.5 * np.mean(np.sum((pred - Y)**2, axis=0))

In [ ]:
def grad_fn(theta, X, Y):
    Ws = unpack(theta)
    N = X.shape[1]
    acts = [X.copy()]
    for W in Ws:
        acts.append(W @ acts[-1])
    delta = (acts[-1] - Y) / N
    grads = []
    for l in range(N_LAYERS - 1, -1, -1):
        grads.insert(0, delta @ acts[l].T)
        if l > 0:
            delta = Ws[l].T @ delta
    return pack(grads), grads

---
## Section 5: Hessian via Finite Differences

The full Hessian $H_{ij} = \frac{\partial^2 \mathcal{L}}{\partial \theta_i \partial \theta_j}$ is computed column-by-column using central finite differences:

$$H_{:,i} \approx \frac{\nabla \mathcal{L}(\theta + \epsilon e_i) - \nabla \mathcal{L}(\theta - \epsilon e_i)}{2\epsilon}$$

This requires $2 \times 48 = 96$ gradient evaluations per Hessian. The result is symmetrized to ensure numerical stability. For our 48-parameter network, this produces a $48 \times 48$ matrix whose eigenvalues reveal the local curvature structure -- positive eigenvalues indicate bowl-like directions, negative eigenvalues indicate saddle directions.

In [ ]:
def hessian_fd(theta, X, Y):
    n = len(theta)
    H = np.zeros((n, n))
    for i in range(n):
        tp = theta.copy(); tp[i] += FD_EPS
        tm = theta.copy(); tm[i] -= FD_EPS
        gp, _ = grad_fn(tp, X, Y)
        gm, _ = grad_fn(tm, X, Y)
        H[:, i] = (gp - gm) / (2 * FD_EPS)
    return 0.5 * (H + H.T)

---
## Section 6: Partial SV Equalization -- The Core Mechanism Under Test

This is the critical function that implements the "partial Muon" step. Given a per-layer gradient matrix $G \in \mathbb{R}^{4 \times 4}$ and equalization depth $k$:

1. **Compute SVD**: $G = U \Sigma V^T$ where $\Sigma = \text{diag}(\sigma_1, \ldots, \sigma_4)$, $\sigma_1 \geq \sigma_2 \geq \sigma_3 \geq \sigma_4$
2. **If k = 0**: Return $G$ unchanged (SGD)
3. **If k = 4**: Return $U V^T$ (polar factor, full Muon)
4. **If 0 < k < 4**: Set $\sigma_1, \ldots, \sigma_k$ to their mean, keep $\sigma_{k+1}, \ldots, \sigma_4$ as-is, then rescale to match polar factor norm $\sqrt{d}$

The normalization step (rescaling to match $\sqrt{d}$) ensures we are comparing the **directional** effect of partial equalization, not conflating it with step-size changes. But this rescaling itself creates a subtle issue: the non-equalized singular values get rescaled by a factor that depends on the equalized ones, introducing an implicit coupling between the two subspaces.

### The Newton-Schulz Alternative

The `newton_schulz_layer` function provides an iterative approximation to the polar factor without explicit SVD. It is included for completeness but is NOT used in the main experiment -- we use the SVD-based partial equalization to have precise control over which singular values are modified.

In [ ]:
def partial_sv_equalize_layer(M, k):
    """Equalize top-k SVs of a single layer gradient matrix."""
    if k == 0:
        return M
    U, sigma, Vt = np.linalg.svd(M, full_matrices=False)
    d = len(sigma)
    kk = min(k, d)
    sigma_new = sigma.copy()

    if kk >= d:
        # Full equalization: all SVs to 1 (polar factor)
        return U @ Vt

    # Partial: equalize top-k to their mean
    top_mean = np.mean(sigma[:kk])
    sigma_new[:kk] = top_mean

    # Normalize to match polar factor norm
    target_norm = np.sqrt(d)
    current_norm = np.linalg.norm(sigma_new)
    if current_norm > 1e-15:
        sigma_new *= target_norm / current_norm

    return U @ np.diag(sigma_new) @ Vt

In [ ]:
def newton_schulz_layer(M, n_iters=5):
    norm = np.linalg.norm(M, 'fro')
    if norm < 1e-15:
        return M
    X = M / norm
    for _ in range(n_iters):
        A = X.T @ X
        X = 1.5 * X - 0.5 * X @ A
    return X

---
## Section 7: Main Experiment -- Hessian Saddle-Point Analysis

The main loop executes the following protocol for each of 5 random seeds:

1. **Initialize** a 3-layer deep linear network near identity (I + 0.1 * noise)
2. **Warm up** for 50 SGD steps with momentum to reach a non-trivial landscape point
3. **Compute baseline Hessian** at the warmup endpoint -- this tells us how many saddle directions exist *before* any equalization step
4. **For each k in {0, 1, 2, 3, 4}**:
   - Compute the gradient at the warmup point
   - Apply partial SV equalization with depth k to each layer's gradient
   - Take one step: $\theta_{\text{new}} = \theta - \alpha \cdot g_{\text{equalized}}$
   - Compute the Hessian at $\theta_{\text{new}}$
   - Count negative eigenvalues (below threshold $-10^{-8}$ to filter numerical noise)
   - Record the minimum eigenvalue (most negative = deepest saddle direction)

The key comparison is: do intermediate k values (1, 2, 3) produce MORE negative eigenvalues than the consistent extremes (k=0 and k=4)?

In [ ]:
def main():
    seeds = [42 + i * 137 for i in range(NUM_SEEDS)]

    print("=" * 100)
    print("H19a: PARTIAL SV EQUALIZATION CREATES SPURIOUS SADDLE POINTS?")
    print("=" * 100)
    print(f"Network: {N_LAYERS}-layer deep linear {DIM}x{DIM} ({N_PARAMS} params)")
    print(f"k per layer: {K_PER_LAYER}")
    print(f"Warmup: {WARMUP_STEPS} SGD steps, then measure Hessian before/after one step")
    print(f"Seeds: {seeds}")
    print(f"Negative eigenvalue threshold: -1e-8 (to filter FD numerical noise)")
    print()

    # Accumulators
    neg_eigs_before = {k: [] for k in K_PER_LAYER}
    neg_eigs_after = {k: [] for k in K_PER_LAYER}
    min_eig_after = {k: [] for k in K_PER_LAYER}

    for si, seed in enumerate(seeds):
        rng = np.random.RandomState(seed)
        X = rng.randn(DIM, 64) * 0.3
        Y = rng.randn(DIM, 64) * 0.3
        weights = [np.eye(DIM) + rng.randn(DIM, DIM) * 0.1 for _ in range(N_LAYERS)]
        theta = pack(weights)

        print(f"\n{'─'*80}")
        print(f"  SEED {si+1}/{NUM_SEEDS} (seed={seed})")
        print(f"{'─'*80}")
        print(f"  Data: X shape {X.shape}, Y shape {Y.shape}, ||X||_F={np.linalg.norm(X):.4f}, ||Y||_F={np.linalg.norm(Y):.4f}")
        
        # Print initial weight singular values to show starting geometry
        for l in range(N_LAYERS):
            svs = np.linalg.svd(weights[l], compute_uv=False)
            print(f"  Init W{l+1} SVs: [{', '.join(f'{s:.4f}' for s in svs)}]  (condition: {svs[0]/svs[-1]:.2f})")
        
        init_loss = loss_fn(theta, X, Y)
        print(f"  Initial loss: {init_loss:.6f}")

        # Warmup with SGD
        mom = np.zeros_like(theta)
        for step in range(WARMUP_STEPS):
            g, _ = grad_fn(theta, X, Y)
            mom = MOMENTUM * mom + g
            theta = theta - LR * mom
        
        warmup_loss = loss_fn(theta, X, Y)
        print(f"  After {WARMUP_STEPS} SGD warmup steps: loss = {warmup_loss:.6f} (reduction: {init_loss - warmup_loss:.6f})")
        
        # Show post-warmup weight geometry
        Ws_warmup = unpack(theta)
        for l in range(N_LAYERS):
            svs = np.linalg.svd(Ws_warmup[l], compute_uv=False)
            print(f"  Post-warmup W{l+1} SVs: [{', '.join(f'{s:.4f}' for s in svs)}]  (condition: {svs[0]/svs[-1]:.2f})")

        # Hessian at warmup point
        print(f"\n  Computing baseline Hessian ({N_PARAMS}x{N_PARAMS} via finite differences)...")
        H_before = hessian_fd(theta, X, Y)
        eigs_before = np.linalg.eigvalsh(H_before)
        n_neg_before = np.sum(eigs_before < -1e-8)

        print(f"  Baseline Hessian eigenvalue spectrum:")
        print(f"    min = {eigs_before[0]:.6e}, max = {eigs_before[-1]:.6e}")
        print(f"    # negative (< -1e-8): {n_neg_before} out of {N_PARAMS}")
        print(f"    # near-zero (|eig| < 1e-8): {np.sum(np.abs(eigs_before) < 1e-8)}")
        print(f"    # positive (> 1e-8): {np.sum(eigs_before > 1e-8)}")
        print(f"    Trace (sum of eigs): {np.sum(eigs_before):.6e}")
        print(f"    Bottom-5 eigenvalues: [{', '.join(f'{e:.4e}' for e in eigs_before[:5])}]")
        print(f"    Top-5 eigenvalues: [{', '.join(f'{e:.4e}' for e in eigs_before[-5:])}]")

        # For each k: take one step, measure Hessian
        print(f"\n  Per-k Hessian analysis after one step:")
        print(f"  {'k':>4}  {'neg eigs':>10}  {'min eig':>14}  {'max eig':>14}  {'trace':>14}  {'step ||.||':>12}  {'new loss':>12}")
        print(f"  {'─'*80}")
        
        for k in K_PER_LAYER:
            theta_k = theta.copy()
            g, grads_list = grad_fn(theta_k, X, Y)

            # Apply partial SV equalization per layer
            step_layers = []
            for l in range(N_LAYERS):
                step_layers.append(partial_sv_equalize_layer(grads_list[l], k))
            step_vec = pack(step_layers)

            theta_new = theta_k - LR * step_vec
            
            # Measure loss at new point
            new_loss = loss_fn(theta_new, X, Y)
            step_norm = np.linalg.norm(LR * step_vec)
            
            H_after = hessian_fd(theta_new, X, Y)
            eigs_after_k = np.linalg.eigvalsh(H_after)
            n_neg_after = np.sum(eigs_after_k < -1e-8)
            min_eig = eigs_after_k[0]

            neg_eigs_before[k].append(n_neg_before)
            neg_eigs_after[k].append(n_neg_after)
            min_eig_after[k].append(min_eig)

            print(f"  {k:>4}  {n_neg_after:>10}  {min_eig:>14.4e}  {eigs_after_k[-1]:>14.4e}  {np.sum(eigs_after_k):>14.4e}  {step_norm:>12.6f}  {new_loss:>12.6f}")
        
        # Print per-layer gradient SV structure for this seed to understand the anisotropy
        print(f"\n  Gradient SV structure at warmup point (this is what gets equalized):")
        g_full, grads_list_diag = grad_fn(theta, X, Y)
        for l in range(N_LAYERS):
            grad_svs = np.linalg.svd(grads_list_diag[l], compute_uv=False)
            ratio = grad_svs[0] / grad_svs[-1] if grad_svs[-1] > 1e-15 else float('inf')
            print(f"    Grad W{l+1} SVs: [{', '.join(f'{s:.4e}' for s in grad_svs)}]  (condition: {ratio:.2f})")

    # =========================================================================
    # RESULTS
    # =========================================================================
    print(f"\n\n{'='*100}")
    print("AGGREGATE RESULTS: NEGATIVE HESSIAN EIGENVALUES AFTER ONE STEP")
    print(f"{'='*100}")

    print(f"\n  {'k':>4}  {'Mean neg eigs (after)':>22}  {'Std':>8}  {'Mean min eig':>16}  {'Std':>14}  {'Mean neg eigs (before)':>24}")
    print("  " + "-" * 90)
    for k in K_PER_LAYER:
        print(f"  {k:>4}  {np.mean(neg_eigs_after[k]):>22.1f}  {np.std(neg_eigs_after[k]):>8.1f}  "
              f"{np.mean(min_eig_after[k]):>16.4e}  {np.std(min_eig_after[k]):>14.4e}  {np.mean(neg_eigs_before[k]):>24.1f}")

    # Per-seed detail table
    print(f"\n  Per-seed negative eigenvalue counts:")
    print(f"  {'Seed':>6}  {'Before':>8}", end="")
    for k in K_PER_LAYER:
        print(f"  {'k='+str(k):>8}", end="")
    print()
    print("  " + "-" * 60)
    for si in range(NUM_SEEDS):
        print(f"  {si+1:>6}  {neg_eigs_before[0][si]:>8}", end="")
        for k in K_PER_LAYER:
            val = neg_eigs_after[k][si]
            # Highlight if worse than both k=0 and k=4 for this seed
            marker = " *" if val > max(neg_eigs_after[0][si], neg_eigs_after[4][si]) else "  "
            print(f"  {val:>6}{marker}", end="")
        print()
    print("  (* = more saddle directions than both k=0 and k=4 for this seed)")

    # Test: intermediate k has more negative eigenvalues than k=0 or k=4
    mean_neg_0 = np.mean(neg_eigs_after[0])
    mean_neg_4 = np.mean(neg_eigs_after[4])
    intermediate_worse = any(
        np.mean(neg_eigs_after[k]) > max(mean_neg_0, mean_neg_4) + 0.5
        for k in [1, 2, 3]
    )

    print(f"\n  === HYPOTHESIS TEST ===")
    print(f"  k=0 (SGD) mean neg eigs: {mean_neg_0:.1f}")
    print(f"  k=4 (full) mean neg eigs: {mean_neg_4:.1f}")
    print(f"  Baseline max(k=0, k=4): {max(mean_neg_0, mean_neg_4):.1f}")
    for k in [1, 2, 3]:
        excess = np.mean(neg_eigs_after[k]) - max(mean_neg_0, mean_neg_4)
        print(f"  k={k} mean neg eigs: {np.mean(neg_eigs_after[k]):.1f}  (excess over baseline: {excess:+.1f})")
    print(f"\n  Intermediate k creates MORE saddle directions: {'CONFIRMED' if intermediate_worse else 'NOT CONFIRMED'}")
    
    if intermediate_worse:
        print(f"\n  INTERPRETATION: Partial SV equalization creates a geometrically inconsistent")
        print(f"  update direction. The equalized SVs push toward isotropic exploration while")
        print(f"  the non-equalized SVs retain the original anisotropy. This mismatch induces")
        print(f"  negative curvature along directions that mix the two subspaces -- saddle")
        print(f"  points that exist in neither the pure SGD nor the pure Muon landscape.")
    else:
        print(f"\n  INTERPRETATION: The saddle-point creation mechanism is not cleanly observed")
        print(f"  in this setting. Possible explanations:")
        print(f"    - The 4x4 network may be too small for the subspace mismatch to manifest")
        print(f"    - The warmup point may already have saturated saddle structure")
        print(f"    - The normalization step may be compensating for the geometric mismatch")
        print(f"    - The single-step measurement may miss accumulation effects over training")

    # Additional diagnostic: minimum eigenvalue trend
    print(f"\n  === MINIMUM EIGENVALUE ANALYSIS ===")
    print(f"  (More negative = deeper/sharper saddle direction)")
    for k in K_PER_LAYER:
        vals = min_eig_after[k]
        print(f"  k={k}: min eig = {np.mean(vals):.4e} +/- {np.std(vals):.4e}  "
              f"[range: {np.min(vals):.4e} to {np.max(vals):.4e}]")

    print(f"\n{'='*100}")
    print("EXPERIMENT COMPLETE")
    print(f"{'='*100}")

In [ ]:
if __name__ == '__main__':
    main()

---
## Section 8: Conclusions and Implications

### What the Results Mean

This experiment probes the **geometric consistency** hypothesis for why partial Muon hurts. The two key metrics are:

1. **Negative eigenvalue count**: How many directions in parameter space are saddle-like (loss decreases in one direction, increases in another). More negative eigenvalues = more saddle structure = harder optimization landscape.

2. **Minimum eigenvalue**: The most negative eigenvalue indicates the *sharpest* saddle direction. A more negative minimum eigenvalue means the optimizer faces a steeper "wrong way" gradient in that direction.

### Why This Matters for Muon Theory

If confirmed, the spurious saddle mechanism provides a **causal** explanation for the H3b finding that partial equalization hurts:

- **Full Muon** (k=4): The polar factor $UV^T$ creates a geometrically consistent update where all singular directions are treated equally. The resulting landscape point has curvature consistent with isotropic exploration.

- **SGD** (k=0): The raw gradient respects the original landscape anisotropy. Steps follow the natural curvature, arriving at points with curvature consistent with that anisotropy.

- **Partial** (k=1,2,3): The step direction is a chimera -- partly isotropic, partly anisotropic. The new landscape point inherits conflicting curvature signals from these two regimes, manifesting as spurious negative eigenvalues.

### Connection to RG Gauge Fixing

In the renormalization group interpretation, full SV equalization corresponds to complete gauge fixing -- removing all redundant degrees of freedom. Partial equalization is an *incomplete* gauge fixing that creates a **gauge artifact**: the optimizer sees phantom directions that look like saddles but are actually artifacts of the inconsistent gauge choice. This is analogous to Gribov copies in non-Abelian gauge theory, where incomplete gauge fixing creates spurious critical points in the Faddeev-Popov determinant.

### Limitations

- The 4x4 network (48 params) may be too small for the subspace mismatch to strongly manifest -- with only 4 singular values per layer, the "boundary" between equalized and non-equalized subspaces is just 1 dimension wide.
- Single-step measurement may miss cumulative saddle creation over many training steps.
- The normalization to $\sqrt{d}$ introduces an implicit coupling that could either amplify or suppress the mismatch effect.